# CASAS Aruba — confronto controllato fra algoritmi e soglie

Questo esperimento cambia **solo algoritmo/parametri**, mantenendo identici dataset, preprocessing e casi di valutazione. Non è un test su giorni nuovi: la discovery usa l'intero log e la valutazione rimane **in-sample**. Un miglioramento qui non dimostra generalizzazione.

**Si parte dal dataset originale `.txt`**, come nel notebook precedente. Caricalo su Drive e indica il percorso nella sezione 3. La sezione 4 applica il preprocessing del progetto una sola volta: tutti e tre gli algoritmi useranno il CSV generato. Per confrontare anche i risultati precedenti, usa lo stesso file raw e la stessa finestra temporale; con dati diversi il confronto con il passato non isola il solo effetto del modello.

Vengono confrontati **7 candidati, appartenenti a tutti e tre gli algoritmi**:

| Algoritmo | Soglie |
|---|---|
| Inductive Miner | rumore 0.20 (riferimento attuale), 0.00, 0.10 |
| Heuristic Miner | dipendenza 0.10, 0.30, 0.50 |
| Alpha Miner | configurazione standard, senza soglie di rumore/dipendenza |

Ogni candidato usa l'intero log per la discovery e gli stessi 20 casi completi selezionati con seed 42 per la valutazione. Se il log ordinato coincide con quello dell'ultima esecuzione e aveva lo stesso campionamento, i casi saranno gli stessi. L'elenco e l'impronta del dataset sono esportati per verificarlo.

Oltre a fitness, precision e media armonica vengono misurati tracce fitting, copertura delle etichette e fitness dopo permutazione delle zone entro i casi. **Fitness alta anche dopo la permutazione segnala scarsa selettività rispetto all'ordine.**

Le configurazioni girano **in processi separati, una alla volta**, con limite iniziale di 10 minuti e 4 GiB di RAM per worker. Il monitor controlla la memoria ogni 0,5 secondi: riduce il rischio ma non può garantire protezione da ogni picco OOM. La RAM totale del runtime deve includere anche notebook e librerie. Le configurazioni interrotte sono registrate come tali, senza punteggi inventati. Puoi modificare i limiti nella sezione 6.

Il modello di ogni candidato viene copiato su Drive prima della valutazione e il confronto viene salvato dopo ogni tentativo. Le metriche potrebbero non essere disponibili se la valutazione viene interrotta.

La classifica ordina la media armonica **in-sample**; nessun candidato viene automaticamente dichiarato scientificamente valido. Alpha Miner è incluso come confronto aggiuntivo, con gli stessi dati, valutazione e limiti di risorse degli altri candidati. [Documentazione algoritmi PM4Py](https://processintelligence.solutions/pm4py/examples/process-discovery/).


## 1. Installa le dipendenze di calcolo
Se Colab richiede un riavvio dopo l’installazione, riavvia il runtime e riparti dalla cella 2.

In [ ]:
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "pm4py==2.7.23.11", "pandas>=2.0,<4", "numpy>=1.26,<3", "psutil"])


## 2. Prepara il codice incorporato

In [ ]:
from pathlib import Path
import sys
PROJECT = Path("/content/aruba_project")
PROJECT.mkdir(parents=True, exist_ok=True)
SOURCES = {'colab_pipeline.py': '"""Strumenti Colab: ETL in streaming e modelli esportabili alla dashboard.\n\nNon modifica la pipeline locale né la semantica della mappa sensori corrente.\n"""\nimport hashlib\nimport importlib.metadata\nimport json\nimport pickle\nimport platform\nimport shutil\nfrom datetime import date, datetime\nfrom pathlib import Path\n\nimport pandas as pd\nimport pm4py\nfrom preprocessing import SENSOR_ACTIVITY_MAP, get_time_of_day\n\nCASE, ACTIVITY, TIME = "case:concept:name", "concept:name", "time:timestamp"\n\n\ndef preprocess_raw_to_csv(source, destination, start_date=None, end_date=None):\n    """Preprocessing del progetto in streaming: M/D, mappa, fasce e compressione.\n\n    Entrambi gli stati sono mantenuti. Il raw deve essere cronologico;\n    le date opzionali includono l\'intera giornata iniziale/finale.\n    """\n    import csv\n    from collections import Counter\n\n    source, destination = Path(source), Path(destination)\n    if source.resolve() == destination.resolve():\n        raise ValueError("Ingresso e uscita devono essere file diversi")\n    lower = date.fromisoformat(start_date) if start_date else None\n    upper = date.fromisoformat(end_date) if end_date else None\n    if lower and upper and lower > upper:\n        raise ValueError("START_DATE deve precedere o coincidere con END_DATE")\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    counts = {\n        "lines": 0, "malformed_lines": 0, "excluded_non_motion_door": 0,\n        "excluded_unmapped": 0, "invalid_timestamps": 0, "outside_date_range": 0,\n        "mapped_events": 0, "written_events": 0,\n    }\n    unmapped, states, zones, periods = Counter(), Counter(), Counter(), Counter()\n    cases = set()\n    first_timestamp = final_timestamp = last_timestamp = None\n    previous = None\n    temporary = destination.with_suffix(".partial.csv")\n    try:\n        with source.open(encoding="utf-8-sig") as incoming, temporary.open("w", encoding="utf-8", newline="") as outgoing:\n            writer = csv.writer(outgoing)\n            writer.writerow([CASE, ACTIVITY, TIME])\n            for line in incoming:\n                counts["lines"] += 1\n                parts = line.split()\n                if len(parts) < 4:\n                    counts["malformed_lines"] += 1\n                    continue\n                if not parts[2].startswith(("M", "D")):\n                    counts["excluded_non_motion_door"] += 1\n                    continue\n                if parts[2] not in SENSOR_ACTIVITY_MAP:\n                    counts["excluded_unmapped"] += 1\n                    unmapped[parts[2]] += 1\n                    continue\n                try:\n                    timestamp = datetime.fromisoformat(parts[0] + " " + parts[1])\n                except ValueError:\n                    counts["invalid_timestamps"] += 1\n                    continue\n                if last_timestamp is not None and timestamp < last_timestamp:\n                    raise ValueError(f"Raw non cronologico alla riga {counts[\'lines\']}: ordinare il file prima della compressione streaming")\n                last_timestamp = timestamp\n                if (lower and timestamp.date() < lower) or (upper and timestamp.date() > upper):\n                    counts["outside_date_range"] += 1\n                    continue\n                first_timestamp = first_timestamp or timestamp\n                final_timestamp = timestamp\n                counts["mapped_events"] += 1\n                states[parts[3]] += 1\n                period = get_time_of_day(timestamp.hour)\n                case = f"{timestamp.date()}_{period}"\n                activity = SENSOR_ACTIVITY_MAP[parts[2]]\n                if (case, activity) != previous:\n                    writer.writerow([case, activity, timestamp.isoformat(sep=" ")])\n                    counts["written_events"] += 1\n                    zones[activity] += 1\n                    periods[period] += 1\n                    cases.add(case)\n                    previous = (case, activity)\n        if not counts["written_events"]:\n            raise ValueError(f"Nessun evento utilizzabile nel raw o nella finestra selezionata. Contatori: {counts}")\n        temporary.replace(destination)\n    except Exception:\n        temporary.unlink(missing_ok=True)\n        raise\n    counts.update({\n        "consecutive_duplicates_removed": counts["mapped_events"] - counts["written_events"],\n        "cases": len(cases), "zones": len(zones),\n        "first_input_event_in_window": first_timestamp.isoformat(sep=" "),\n        "last_input_event_in_window": final_timestamp.isoformat(sep=" "),\n        "requested_start_date": start_date, "requested_end_date": end_date,\n        "states_before_compression": dict(sorted(states.items())),\n        "unmapped_sensor_counts": dict(sorted(unmapped.items())),\n        "events_by_zone": dict(sorted(zones.items())),\n        "events_by_period": dict(sorted(periods.items())),\n        "sensor_activity_map": dict(SENSOR_ACTIVITY_MAP),\n        "case_definition": "date_plus_period_00_06_12_19_24",\n        "state_filter": "all_states_as_in_local_project",\n    })\n    return counts\n\n\ndef load_event_log(path):\n    # Carica solo le tre colonne necessarie. Il log derivato deve comunque stare in RAM.\n    df = pd.read_csv(path, usecols=[CASE, ACTIVITY, TIME], dtype={CASE: "string", ACTIVITY: "string"})\n    if df.empty or df[[CASE, ACTIVITY, TIME]].isna().any().any():\n        raise ValueError("Event log vuoto o con colonne PM4Py mancanti/nulle")\n    df[TIME] = pd.to_datetime(df[TIME], format="ISO8601", errors="raise")\n    df = df.sort_values(TIME, kind="stable").reset_index(drop=True)\n    # Condividi le stringhe ripetute senza utilizzare categorie incompatibili con alcune versioni PM4Py.\n    for col in (CASE, ACTIVITY):\n        codes, values = pd.factorize(df[col], sort=False)\n        df[col] = values.astype(object).take(codes)\n    return df[[CASE, ACTIVITY, TIME]]\n\n\ndef discover_one(df, miner="Inductive Miner", noise_threshold=0.2, dependency_threshold=0.5):\n    common = dict(case_id_key=CASE, activity_key=ACTIVITY, timestamp_key=TIME)\n    if miner == "Inductive Miner":\n        model = pm4py.discover_petri_net_inductive(df, noise_threshold=noise_threshold, **common)\n    elif miner == "Heuristic Miner":\n        model = pm4py.discover_petri_net_heuristics(df, dependency_threshold=dependency_threshold, **common)\n    elif miner == "Alpha Miner":\n        model = pm4py.discover_petri_net_alpha(df, **common)\n    else:\n        raise ValueError(f"Miner non supportato: {miner}")\n    return {miner: model}\n\n\ndef sha256_file(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as source:\n        for block in iter(lambda: source.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef export_results(df, models, directory, source_path, settings, scores=None, evaluation_metadata=None):\n    root = Path(directory)\n    data = root / "data"\n    data.mkdir(parents=True, exist_ok=True)\n    df[[CASE, ACTIVITY, TIME]].to_csv(data / "dataset_aruba_attivita.csv", index=False)\n    with (data / "models.pkl").open("wb") as handle:\n        pickle.dump(models, handle, protocol=pickle.HIGHEST_PROTOCOL)\n    (data / "metrics.json").write_text(json.dumps(scores or {}, indent=2), encoding="utf-8")\n    if evaluation_metadata is not None:\n        target = data / "evaluation_metadata.json"\n        if Path(evaluation_metadata).resolve() != target.resolve():\n            shutil.copyfile(evaluation_metadata, target)\n    else:\n        (data / "evaluation_metadata.json").unlink(missing_ok=True)\n    for i, model in enumerate(models.values(), start=1):\n        pm4py.write_pnml(*model, str(data / f"model_{i}.pnml"))\n    versions = {name: importlib.metadata.version(name) for name in ["pm4py", "pandas", "numpy"]}\n    manifest = {\n        "python": platform.python_version(), "packages": versions,\n        "source_name": Path(source_path).name, "source_sha256": sha256_file(source_path),\n        "events": len(df), "cases": int(df[CASE].nunique()), "activities": int(df[ACTIVITY].nunique()),\n        "settings": settings, "evaluation": "completed" if scores is not None else "not_run",\n        "files_sha256": {str(p.relative_to(root)): sha256_file(p) for p in data.iterdir() if p.is_file()},\n    }\n    (root / "run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")\n    (root / "requirements_colab.txt").write_text(\n        "".join(f"{name}=={version}\\n" for name, version in versions.items()), encoding="utf-8")\n    return root\n\n\ndef package_results(directory):\n    directory = Path(directory)\n    return Path(shutil.make_archive(str(directory), "zip", root_dir=directory))\n', 'colab_benchmark.py': '"""Confronti PM4Py isolati, sullo stesso log e sugli stessi casi in-sample."""\nimport gc\nimport hashlib\nimport importlib.metadata\nimport json\nimport os\nimport pickle\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport pm4py\nfrom colab_pipeline import CASE, ACTIVITY, TIME, load_event_log, discover_one, sha256_file\nfrom evaluation import f1_score\n\nDEFAULT_CONFIGS = [\n    {"id": "inductive_020", "label": "Inductive noise=0.20 (riferimento)", "miner": "Inductive Miner", "parameters": {"noise_threshold": .2}},\n    {"id": "inductive_000", "label": "Inductive noise=0.00", "miner": "Inductive Miner", "parameters": {"noise_threshold": .0}},\n    {"id": "inductive_010", "label": "Inductive noise=0.10", "miner": "Inductive Miner", "parameters": {"noise_threshold": .1}},\n    {"id": "heuristic_030", "label": "Heuristic dependency=0.30", "miner": "Heuristic Miner", "parameters": {"dependency_threshold": .3}},\n    {"id": "heuristic_050", "label": "Heuristic dependency=0.50", "miner": "Heuristic Miner", "parameters": {"dependency_threshold": .5}},\n    {"id": "heuristic_010", "label": "Heuristic dependency=0.10", "miner": "Heuristic Miner", "parameters": {"dependency_threshold": .1}},\n    {"id": "alpha", "label": "Alpha Miner", "miner": "Alpha Miner", "parameters": {}},\n]\n\n\ndef write_json(path, value):\n    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding="utf-8")\n\n\ndef prepare_evaluation(csv_path, output_path, max_cases=20, seed=42):\n    if max_cases < 1:\n        raise ValueError("max_cases deve essere positivo")\n    df = load_event_log(csv_path)\n    cases = df[CASE].unique()\n    selected = np.random.default_rng(seed).choice(cases, max_cases, replace=False) if len(cases) > max_cases else cases\n    sample = df[df[CASE].isin(selected)].copy()\n    sample.to_csv(output_path, index=False, lineterminator="\\n")\n    fingerprint = df[[CASE, ACTIVITY, TIME]].copy()\n    fingerprint[TIME] = fingerprint[TIME].astype(str)\n    digest = hashlib.sha256(fingerprint.to_csv(index=False, lineterminator="\\n").encode()).hexdigest()\n    metadata = {\n        "seed": seed, "max_eval_cases": max_cases, "total_cases": int(len(cases)),\n        "evaluated_cases": sample[CASE].unique().tolist(), "evaluated_events": len(sample),\n        "dataset_sha256": digest, "pm4py_version": pm4py.__version__,\n        "selection_criterion": "harmonic_mean_fitness_precision",\n        "scope": "in_sample: stesso log di discovery, stessi casi e seed per tutti i candidati",\n    }\n    del fingerprint, sample, df\n    gc.collect()\n    return metadata\n\n\ndef worker(spec_path):\n    spec = json.loads(Path(spec_path).read_text(encoding="utf-8"))\n    directory = Path(spec["directory"])\n    config = spec["config"]\n    started = time.monotonic()\n    try:\n        df = load_event_log(spec["full_csv"])\n        log_labels = set(df[ACTIVITY])\n        model = discover_one(df, miner=config["miner"], **config["parameters"])[config["miner"]]\n        # Salva prima della valutazione: resta recuperabile se precision esaurisce le risorse.\n        with (directory / "models.pkl").open("wb") as handle:\n            pickle.dump({config["label"]: model}, handle, protocol=pickle.HIGHEST_PROTOCOL)\n        pm4py.write_pnml(*model, str(directory / "model.pnml"))\n        if spec.get("checkpoint_dir"):\n            checkpoint = Path(spec["checkpoint_dir"])\n            checkpoint.mkdir(parents=True, exist_ok=True)\n            shutil.copy2(directory / "models.pkl", checkpoint / "models.pkl")\n            shutil.copy2(directory / "model.pnml", checkpoint / "model.pnml")\n            write_json(checkpoint / "discovery.json", {"stage": "discovery_completed", "config": config})\n        del df\n        gc.collect()\n        sample = load_event_log(spec["evaluation_csv"])\n        fitness = pm4py.fitness_token_based_replay(sample, *model)\n        precision = pm4py.precision_token_based_replay(sample, *model)\n        metrics = {"fitness": float(fitness["log_fitness"]), "precision": float(precision)}\n        shuffled = sample[[CASE, ACTIVITY, TIME]].copy()\n        rng = np.random.default_rng(spec["seed"])\n        for _, group in shuffled.groupby(CASE, sort=False):\n            shuffled.loc[group.index, ACTIVITY] = rng.permutation(group[ACTIVITY].to_numpy())\n        shuffled_fitness = pm4py.fitness_token_based_replay(shuffled, *model)\n        net = model[0]\n        visible = {t.label for t in net.transitions if t.label is not None}\n        result = {\n            "status": "ok", "id": config["id"], "label": config["label"],\n            "metrics": metrics, "f1": f1_score(metrics),\n            "fit_traces_percent": fitness["percentage_of_fitting_traces"],\n            "shuffled_fitness": shuffled_fitness["log_fitness"],\n            "shuffled_fit_traces_percent": shuffled_fitness["percentage_of_fitting_traces"],\n            "order_fitness_gap": fitness["log_fitness"] - shuffled_fitness["log_fitness"],\n            "places": len(net.places), "transitions": len(net.transitions), "arcs": len(net.arcs),\n            "silent_transitions": sum(t.label is None for t in net.transitions),\n            "missing_log_labels": sorted(log_labels - visible),\n            "evaluated_cases": sample[CASE].unique().tolist(),\n            "seconds": time.monotonic() - started,\n        }\n        write_json(directory / "result.json", result)\n        return 0\n    except Exception as exc:\n        write_json(directory / "result.json", {"status": "error", "id": config["id"],\n                   "label": config["label"], "error": f"{type(exc).__name__}: {exc}",\n                   "seconds": time.monotonic() - started})\n        return 1\n\n\ndef process_rss(pid):\n    import psutil\n    try:\n        process = psutil.Process(pid)\n        return process.memory_info().rss + sum(p.memory_info().rss for p in process.children(recursive=True))\n    except psutil.NoSuchProcess:\n        return 0\n\n\ndef monitor_worker(command, log_path, timeout_seconds, max_memory_gb):\n    """Limiti osservati ogni 0,5 s: riducono il rischio, non garantiscono contro picchi OOM."""\n    started = time.monotonic()\n    peak = 0\n    status = None\n    env = dict(os.environ, OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1")\n    flags = subprocess.CREATE_NO_WINDOW if os.name == "nt" else 0\n    with Path(log_path).open("w", encoding="utf-8") as log:\n        with subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, env=env,\n                              creationflags=flags) as process:\n            try:\n                while process.poll() is None:\n                    if time.monotonic() - started > timeout_seconds:\n                        status = "timeout"\n                        break\n                    if max_memory_gb is not None:\n                        peak = max(peak, process_rss(process.pid))\n                        if peak > max_memory_gb * 2**30:\n                            status = "memory_limit"\n                            break\n                    time.sleep(.5)\n            finally:\n                if process.poll() is None:\n                    process.kill()\n                process.wait()\n    return {"status": status or ("finished" if process.returncode == 0 else "worker_error"),\n            "returncode": process.returncode, "peak_worker_gb": peak / 2**30,\n            "seconds": time.monotonic() - started}\n\n\ndef validate_configs(configs):\n    if not configs:\n        raise ValueError("Scegli almeno un candidato")\n    if len({c["id"] for c in configs}) != len(configs) or len({c["label"] for c in configs}) != len(configs):\n        raise ValueError("ID ed etichette devono essere unici")\n    for c in configs:\n        if not re.fullmatch(r"[a-z0-9_-]+", c["id"]):\n            raise ValueError("ID candidato non valido")\n        allowed = {"Inductive Miner": {"noise_threshold"}, "Heuristic Miner": {"dependency_threshold"},\n                   "Alpha Miner": set()}\n        if c["miner"] not in allowed or set(c["parameters"]) != allowed[c["miner"]]:\n            raise ValueError("Algoritmo o parametri non supportati nel confronto")\n        if not all(isinstance(v, (int, float)) and 0 <= v <= 1 for v in c["parameters"].values()):\n            raise ValueError("Le soglie devono essere numeri fra 0 e 1")\n\n\ndef save_checkpoint(root, results, configs, metadata, backup_dir=None):\n    models, scores = {}, {}\n    for config in configs:\n        directory = root / "candidates" / config["id"]\n        if (directory / "models.pkl").exists():\n            with (directory / "models.pkl").open("rb") as handle:\n                models.update(pickle.load(handle))\n        result = next((r for r in results if r["id"] == config["id"]), None)\n        if result is not None:\n            scores[config["label"]] = result["metrics"] if result["status"] == "ok" else {"error": result.get("error", result["status"])}\n    with (root / "data/models.pkl").open("wb") as handle:\n        pickle.dump(models, handle, protocol=pickle.HIGHEST_PROTOCOL)\n    write_json(root / "data/metrics.json", scores)\n    eval_metadata = dict(metadata)\n    eval_metadata["metrics_sha256"] = hashlib.sha256(json.dumps(scores, sort_keys=True).encode()).hexdigest()\n    write_json(root / "data/evaluation_metadata.json", eval_metadata)\n    write_json(root / "comparison_results.json", results)\n    valid = [r for r in results if r["status"] == "ok" and r.get("f1") is not None]\n    winner = max(valid, key=lambda r: r["f1"])["label"] if valid else None\n    write_json(root / "comparison_manifest.json", {\n        "configs": configs, "completed": len(results), "planned": len(configs),\n        "packages": {name: importlib.metadata.version(name) for name in ["pm4py", "pandas", "numpy"]},\n        "first_by_in_sample_f1": winner, "dataset_csv_sha256": sha256_file(root / "data/dataset_aruba_attivita.csv"),\n        "evaluation": eval_metadata,\n        "note": "Classifica in-sample: non dimostra generalizzazione. Controllare anche precision e sequenze permutate.",\n    })\n    (root / "requirements_colab.txt").write_text("".join(f"{name}=={importlib.metadata.version(name)}\\n" for name in ["pm4py", "pandas", "numpy"]), encoding="utf-8")\n    rows = [{"Modello": r["label"], "Stato": r["status"],\n             "Fitness": r.get("metrics", {}).get("fitness"), "Precision": r.get("metrics", {}).get("precision"),\n             "F1": r.get("f1"), "Tracce fitting %": r.get("fit_traces_percent"),\n             "Fitness mescolata": r.get("shuffled_fitness"), "Differenza ordine": r.get("order_fitness_gap"),\n             "Etichette omesse": ", ".join(r.get("missing_log_labels", [])), "Errore": r.get("error", "")}\n            for r in results]\n    pd.DataFrame(rows).to_csv(root / "comparison.csv", index=False)\n    if backup_dir is not None:\n        shutil.copytree(root, backup_dir, dirs_exist_ok=True)\n    del models\n    gc.collect()\n\n\ndef run_comparison(csv_path, output_dir, configs=None, max_eval_cases=20, seed=42,\n                   timeout_seconds=600, max_memory_gb=4, backup_dir=None):\n    configs = DEFAULT_CONFIGS if configs is None else configs\n    validate_configs(configs)\n    if timeout_seconds <= 0 or (max_memory_gb is not None and max_memory_gb <= 0):\n        raise ValueError("Limiti di tempo e memoria non validi")\n    if max_memory_gb is not None:\n        process_rss(os.getpid())  # Richiede psutil, installato dal notebook.\n    root = Path(output_dir).resolve()\n    if backup_dir is not None:\n        backup_dir = Path(backup_dir).resolve()\n        if root == backup_dir or root in backup_dir.parents or backup_dir in root.parents:\n            raise ValueError("Output e backup devono essere cartelle separate, non annidate")\n    root.mkdir(parents=True, exist_ok=False)\n    (root / "data").mkdir()\n    (root / "candidates").mkdir()\n    full_csv = root / "data/dataset_aruba_attivita.csv"\n    shutil.copyfile(csv_path, full_csv)\n    eval_csv = root / "evaluation_log.csv"\n    metadata = prepare_evaluation(full_csv, eval_csv, max_cases=max_eval_cases, seed=seed)\n    results = []\n    save_checkpoint(root, results, configs, metadata, backup_dir)\n    for config in configs:\n        print(f"Avvio {config[\'label\']} (timeout {timeout_seconds}s, RAM worker {max_memory_gb} GiB)", flush=True)\n        directory = root / "candidates" / config["id"]\n        directory.mkdir()\n        spec = {"full_csv": str(full_csv), "evaluation_csv": str(eval_csv),\n                "directory": str(directory), "config": config, "seed": seed,\n                "checkpoint_dir": str(backup_dir / "candidates" / config["id"]) if backup_dir else None}\n        write_json(directory / "spec.json", spec)\n        execution = monitor_worker([sys.executable, str(Path(__file__).resolve()), "--worker", str(directory / "spec.json")],\n                                   directory / "worker.log", timeout_seconds, max_memory_gb)\n        result_path = directory / "result.json"\n        if execution["status"] == "finished" and result_path.exists():\n            result = json.loads(result_path.read_text(encoding="utf-8"))\n        elif execution["status"] == "worker_error" and result_path.exists():\n            result = json.loads(result_path.read_text(encoding="utf-8"))\n        else:\n            result = {"id": config["id"], "label": config["label"], "status": execution["status"],\n                      "error": "Limite tempo/RAM o arresto del worker; consultare worker.log"}\n        result["execution"] = execution\n        results.append(result)\n        save_checkpoint(root, results, configs, metadata, backup_dir)\n        print(f"  -> {result[\'status\']}; F1={result.get(\'f1\')}; backup aggiornato", flush=True)\n    return pd.read_csv(root / "comparison.csv")\n\n\nif __name__ == "__main__":\n    if len(sys.argv) == 3 and sys.argv[1] == "--worker":\n        raise SystemExit(worker(sys.argv[2]))\n    raise SystemExit("Usare run_comparison dal notebook")\n', 'preprocessing.py': 'import pandas as pd\nimport os\nimport pm4py\n\n# ==========================================\n# CONFIGURAZIONE COSTANTI\n# ==========================================\nCASE_ID_KEY = "case:concept:name"\nACTIVITY_KEY = "concept:name"\nTIMESTAMP_KEY = "time:timestamp"\n\n# ==========================================\n# 1. MAPPA ATTIVITÀ\n# ==========================================\nSENSOR_ACTIVITY_MAP = {\n    # ZONA NOTTE\n    "M001": "Camera_Letto", "M002": "Camera_Letto", \n    "M003": "Camera_Letto", "M004": "Camera_Letto",\n    "M005": "Bagno_Principale", "M006": "Bagno_Principale", "M007": "Bagno_Principale",\n    \n    # ZONA GIORNO\n    "M008": "Soggiorno", "M009": "Soggiorno", "M010": "Soggiorno", \n    "M011": "Soggiorno", "M012": "Soggiorno", "M013": "Soggiorno", \n    "M014": "Sala_Pranzo",\n    \n    # CUCINA\n    "M015": "Cucina_Prep", "M016": "Cucina_Prep", "M017": "Cucina_Prep", \n    "M018": "Cucina_Prep", "M019": "Cucina_Prep",\n    \n    # UFFICIO\n    "M020": "Ufficio", "M021": "Ufficio", "M022": "Ufficio",\n    \n    # CORRIDOI\n    "M023": "Corridoio", "M024": "Corridoio",\n    \n    # SECONDA ZONA\n    "M025": "Camera_2", "M026": "Bagno_2",\n    "M027": "Camera_2", "M028": "Bagno_2", \n    "M029": "Camera_2", "M030": "Camera_2", "M031": "Camera_2",\n    \n    # PORTE\n    "D001": "Uscita_Principale", "D002": "Uscita_Retro", "D004": "Garage"\n}\n\ndef get_time_of_day(hour):\n    if 0 <= hour < 6: return "Notte"\n    elif 6 <= hour < 12: return "Mattina"\n    elif 12 <= hour < 19: return "Pomeriggio"\n    else: return "Sera"\n\n# ==========================================\n# 2. CARICAMENTO E PULIZIA\n# ==========================================\ndef load_and_transform_dataset(file_path):\n    print(f"1. Caricamento file: {file_path}")\n    \n    if not os.path.exists(file_path):\n        print(f"ERRORE: File \'{file_path}\' mancante.")\n        # Invece di sys.exit, solleviamo un\'eccezione per essere più pythonic in un modulo\n        raise FileNotFoundError(f"File \'{file_path}\' mancante.")\n\n    data = []\n    try:\n        with open(file_path, \'r\') as f:\n            for line in f:\n                parts = line.strip().split()\n                if len(parts) >= 4:\n                    data.append(parts[:4])\n    except Exception as e:\n        print(f"Errore lettura: {e}")\n        raise e\n\n    df = pd.DataFrame(data, columns=["date", "time", "sensor_id", "value"])\n\n    # Manteniamo entrambi gli stati ON/OFF e OPEN/CLOSE per compatibilità\n    # con il dataset derivato; le etichette identificano zone, non azioni validate.\n    # Filtro Sensori (Solo Motion e Door, NO Temperatura)\n    print("   Filtro Sensori (Eliminazione Temperatura)...")\n    df = df[df[\'sensor_id\'].str.startswith((\'M\', \'D\'))].copy()\n\n    # Timestamp\n    df[\'temp_ts\'] = df[\'date\'] + \' \' + df[\'time\']\n    df[TIMESTAMP_KEY] = pd.to_datetime(df[\'temp_ts\'], format="%Y-%m-%d %H:%M:%S.%f", errors=\'coerce\')\n    df = df.dropna(subset=[TIMESTAMP_KEY]).sort_values(TIMESTAMP_KEY, kind="stable")\n\n    # Mappatura Attività\n    print("   Conversione Sensori -> Attività Umane...")\n    df[\'activity_raw\'] = df[\'sensor_id\'].map(SENSOR_ACTIVITY_MAP)\n    df[\'activity_raw\'] = df[\'activity_raw\'].fillna("Altro")\n    df = df[df[\'activity_raw\'] != "Altro"].copy()\n\n    # Suddivisione Temporale\n    df[\'hour\'] = df[TIMESTAMP_KEY].dt.hour\n    df[\'period\'] = df[\'hour\'].apply(get_time_of_day)\n    df[CASE_ID_KEY] = df[TIMESTAMP_KEY].dt.date.astype(str) + "_" + df[\'period\']\n\n    # Compressione (Rimuove duplicati consecutivi)\n    print(f"   Eventi grezzi: {len(df)}")\n    mask = (df[\'activity_raw\'] != df[\'activity_raw\'].shift(1)) | \\\n           (df[CASE_ID_KEY] != df[CASE_ID_KEY].shift(1))\n    df = df[mask].copy()\n    \n    df[ACTIVITY_KEY] = df[\'activity_raw\']\n    # Mantieni colonne utili\n    final_df = df[[CASE_ID_KEY, ACTIVITY_KEY, TIMESTAMP_KEY]].reset_index(drop=True)\n    \n    print(f"   Eventi finali ottimizzati: {len(final_df)}")\n    return final_df\n\ndef extract_process_structure(df):\n    """\n    Estrae il DFG e le varianti.\n    """\n    if df is None or df.empty:\n        return "Nessun dato disponibile."\n    try:\n        # 1. DFG\n        dfg, start_activities, end_activities = pm4py.discover_dfg(df)\n        sorted_dfg = sorted(dfg.items(), key=lambda x: x[1], reverse=True)\n\n        structure_str = "--- STRUTTURA DEL PROCESSO (DFG) ---\\n"\n        structure_str += f"Start: {list(start_activities.keys())}\\nEnd: {list(end_activities.keys())}\\n"\n        structure_str += "FLUSSI PRINCIPALI (A -> B [Freq]):\\n"\n        for (act_from, act_to), count in sorted_dfg[:30]:\n            structure_str += f"- {act_from} -> {act_to} [{count}]\\n"\n\n        # 2. Varianti (con fix robustezza)\n        variants = pm4py.get_variants_as_tuples(df)\n\n        def get_count(val):\n            return val if isinstance(val, int) else len(val)\n\n        sorted_variants = sorted(variants.items(), key=lambda x: get_count(x[1]), reverse=True)\n\n        structure_str += "\\n--- VARIANTI TOP 5 ---\\n"\n        for path, val in sorted_variants[:5]:\n            count = get_count(val)\n            path_str = " -> ".join(path[:20])\n            if len(path) > 20:\n                path_str += f" -> ... ({len(path)} eventi nella traccia completa)"\n            structure_str += f"- ({count} casi): {path_str}\\n"\n\n        return structure_str\n    except Exception as e:\n        return f"Errore estrazione struttura: {str(e)}"\n', 'evaluation.py': 'import json\nimport math\nfrom hashlib import sha256\nimport numpy as np\nimport pm4py\n# Costanti PM4Py\nCASE_ID_KEY = "case:concept:name"\nACTIVITY_KEY = "concept:name"\nTIMESTAMP_KEY = "time:timestamp"\nRANDOM_SEED = 42\nMAX_EVAL_CASES = 50  # Numero di periodi da valutare (Safe Mode)\n\ndef evaluate(df, net, im, fm):\n    """\n    Valuta un singolo modello su un DataFrame (già convertito in log internamente da PM4Py se necessario).\n    """\n    try:\n        # Replay token-based (Il migliore per performance)\n        # Nota: PM4Py accetta DataFrame direttamente in molte funzioni recenti, \n        # ma assicuriamoci di passare le chiavi corrette.\n        fitness = pm4py.fitness_token_based_replay(\n            df, net, im, fm, \n            case_id_key=CASE_ID_KEY, \n            activity_key=ACTIVITY_KEY, \n            timestamp_key=TIMESTAMP_KEY\n        )\n        precision = pm4py.precision_token_based_replay(\n            df, net, im, fm, \n            case_id_key=CASE_ID_KEY, \n            activity_key=ACTIVITY_KEY, \n            timestamp_key=TIMESTAMP_KEY\n        )\n        # Generalization e Simplicity potrebbero essere pesanti, li includiamo se richiesto o li omettiamo per velocità\n        # Per ora manteniamo Fitness e Precision che sono i più critici.\n        \n        return {\n            "fitness": fitness[\'log_fitness\'], \n            "precision": precision\n        }\n    except Exception as e:\n        return {"error": str(e)}\n\ndef evaluate_models_safe(df, models, metadata_path=None):\n    print("\\n3. Valutazione (Safe Mode)...")\n    unique_cases = df[CASE_ID_KEY].unique()\n    \n    if len(unique_cases) > MAX_EVAL_CASES:\n        sampled_cases = np.random.default_rng(RANDOM_SEED).choice(unique_cases, MAX_EVAL_CASES, replace=False)\n        eval_df = df[df[CASE_ID_KEY].isin(sampled_cases)].copy()\n        print(f"   Valutazione su campione di {MAX_EVAL_CASES} periodi.")\n    else:\n        eval_df = df\n        print(f"   Valutazione su tutti i {len(unique_cases)} periodi.")\n\n    if metadata_path is not None:\n        import pandas as pd\n        fingerprint_df = df[[CASE_ID_KEY, ACTIVITY_KEY, TIMESTAMP_KEY]].copy()\n        fingerprint_df[TIMESTAMP_KEY] = pd.to_datetime(fingerprint_df[TIMESTAMP_KEY]).astype(str)\n        metadata = {\n            "seed": RANDOM_SEED,\n            "max_eval_cases": MAX_EVAL_CASES,\n            "total_cases": int(len(unique_cases)),\n            "evaluated_cases": [str(c) for c in eval_df[CASE_ID_KEY].unique()],\n            "evaluated_events": int(len(eval_df)),\n            "dataset_sha256": sha256(fingerprint_df.to_csv(index=False).encode("utf-8")).hexdigest(),\n            "pm4py_version": pm4py.__version__,\n            "selection_criterion": "harmonic_mean_fitness_precision",\n            "scope": "in_sample: modelli scoperti sull\'intero log, valutati su un sottoinsieme"\n        }\n\n    scores = {}\n    for name, model in models.items():\n        if isinstance(model, dict) and "error" in model:\n            scores[name] = model\n            continue\n        \n        print(f"   Valutando {name}...", end=" ")\n        try:\n            net, im, fm = model\n            res = evaluate(eval_df, net, im, fm)\n            scores[name] = res\n            \n            if "error" not in res:\n                print(f"-> OK (F: {res[\'fitness\']:.2f}, P: {res[\'precision\']:.2f})")\n            else:\n                print(f"-> FALLITO ({res[\'error\']})")\n                \n        except Exception as e:\n            print(f"-> FALLITO (Eccezione imprevista)")\n            scores[name] = {"error": str(e)}\n            \n    if metadata_path is not None:\n        metadata["metrics_sha256"] = sha256(json.dumps(scores, sort_keys=True).encode("utf-8")).hexdigest()\n        with open(metadata_path, "w", encoding="utf-8") as handle:\n            json.dump(metadata, handle, indent=2, ensure_ascii=False)\n    return scores\n\ndef f1_score(metrics):\n    """Media armonica di fitness e precision; rifiuta metriche incomplete/non valide."""\n    if not isinstance(metrics, dict) or "error" in metrics:\n        return None\n    try:\n        fitness = float(metrics["fitness"])\n        precision = float(metrics["precision"])\n    except (KeyError, TypeError, ValueError):\n        return None\n    if not all(math.isfinite(v) and 0 <= v <= 1 for v in (fitness, precision)):\n        return None\n    return 2 * fitness * precision / (fitness + precision) if fitness + precision else 0.0\n\n\ndef choose_best(results: dict):\n    """Seleziona il modello con la maggiore media armonica fitness/precision."""\n    valid = {name: m for name, m in results.items() if f1_score(m) is not None}\n    if not valid:\n        return None, None\n    name = max(valid, key=lambda key: f1_score(valid[key]))\n    return name, valid[name]\n\n\ndef choose_best_model_internal(scores):\n    """Alias compatibile per la dashboard; stesso criterio di choose_best."""\n    return choose_best(scores)[0]\n', 'project_paths.py': '"""Percorsi indipendenti dalla directory di avvio."""\nfrom pathlib import Path\n\nPROJECT_DIR = Path(__file__).resolve().parent\nDATA_DIR = PROJECT_DIR / "data"\nCONFIG_PATH = PROJECT_DIR / "config.json"\nDATASET_PATH = DATA_DIR / "aruba.20111101-20111128_20260218.180945.txt"\nOUTPUT_CSV = DATA_DIR / "dataset_aruba_attivita.csv"\nOUTPUT_MODELS = DATA_DIR / "models.pkl"\nOUTPUT_METRICS = DATA_DIR / "metrics.json"\nEVALUATION_METADATA = DATA_DIR / "evaluation_metadata.json"\n'}
for name, content in SOURCES.items():
    (PROJECT / name).write_text(content, encoding="utf-8")
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
print("Codice disponibile in", PROJECT)


## 3. Collega Drive e configura il file di ingresso
Carica il dataset originale `.txt` nella cartella `ArubaColab/input` di Drive. Modifica **DATASET** con il percorso e il nome esatti del tuo file; `aruba.txt` sotto indica il nome di esempio.


In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import json
import shutil
import gc
import psutil
import pm4py
from colab_pipeline import (preprocess_raw_to_csv, load_event_log, discover_one,
                           export_results, package_results)

drive.mount("/content/drive")

DATASET = Path("/content/drive/MyDrive/ArubaColab/input/aruba.txt")  # MODIFICA QUI
INPUT_KIND = "raw_sensors"  # dataset originale .txt: applica il preprocessing del progetto
# Finestra facoltativa: None elabora tutto il file, date YYYY-MM-DD incluse.
START_DATE = None
END_DATE = None

# Stesso campione per tutti i candidati; come nell’ultima valutazione Colab.
MAX_EVAL_CASES = 20

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_DIR = Path("/content/aruba_runs") / RUN_ID
DRIVE_DIR = Path("/content/drive/MyDrive/ArubaColab/results") / RUN_ID
PREPARED_CSV = RUN_DIR / "data/dataset_aruba_attivita.csv"
RUN_DIR.mkdir(parents=True, exist_ok=False)
PREPARED_CSV.parent.mkdir(exist_ok=True)

if not DATASET.is_file():
    raise FileNotFoundError(f"File non trovato: {DATASET}. Modifica DATASET e riesegui la cella.")
print("RAM totale del runtime:", round(psutil.virtual_memory().total / 2**30, 2), "GiB")
print("Output persistente:", DRIVE_DIR)


## 4. Preprocessing del dataset TXT

Con `INPUT_KIND="raw_sensors"` il file viene letto riga per riga e trasformato con la stessa logica del progetto: filtro dei sensori, associazione alle zone, suddivisione per giorno e fascia oraria, rimozione delle ripetizioni consecutive. Il CSV risultante viene creato automaticamente e condiviso da tutti i candidati.

Lascia `START_DATE=END_DATE=None` per usare tutto il file, oppure imposta le date per scegliere una finestra comune a tutti gli algoritmi. Il report del preprocessing e il CSV vengono salvati anche su Drive.


In [ ]:
import pandas as pd
from preprocessing import SENSOR_ACTIVITY_MAP

# Mostra la mappa effettivamente applicata, identica al progetto locale.
if INPUT_KIND == "raw_sensors":
    display(pd.DataFrame(sorted(SENSOR_ACTIVITY_MAP.items()), columns=["Sensore", "Zona del progetto"]))
    etl_info = preprocess_raw_to_csv(
        DATASET, PREPARED_CSV, start_date=START_DATE, end_date=END_DATE)
    print("Righe lette:", f"{etl_info['lines']:,}")
    print("Eventi mappati prima della compressione:", f"{etl_info['mapped_events']:,}")
    print("Duplicati consecutivi rimossi:", f"{etl_info['consecutive_duplicates_removed']:,}")
    print("Eventi finali:", f"{etl_info['written_events']:,}")
    print("Casi:", etl_info["cases"], "— Zone:", etl_info["zones"])
    print("Periodo degli eventi raw selezionati:", etl_info["first_input_event_in_window"],
          "→", etl_info["last_input_event_in_window"])
    exclusions = {key: etl_info[key] for key in ["malformed_lines", "excluded_non_motion_door",
                 "excluded_unmapped", "invalid_timestamps", "outside_date_range"]}
    print("Esclusioni:", exclusions)
    if etl_info["unmapped_sensor_counts"]:
        print("ATTENZIONE — sensori M/D non mappati:", etl_info["unmapped_sensor_counts"])
    display(pd.DataFrame(list(etl_info["events_by_period"].items()), columns=["Fascia", "Eventi compressi"]))
elif INPUT_KIND == "event_log_csv":
    if START_DATE or END_DATE:
        raise ValueError("START_DATE/END_DATE sono disponibili solo per raw_sensors; filtra prima il CSV elaborato")
    shutil.copyfile(DATASET, PREPARED_CSV)
    etl_info = {"mode": "preformatted_event_log", "preprocessing_applied": False}
    print("CSV già elaborato copiato; la mappa sensori non viene riapplicata.")
else:
    raise ValueError("INPUT_KIND non riconosciuto")

(RUN_DIR / "preprocessing_report.json").write_text(
    json.dumps(etl_info, indent=2, ensure_ascii=False), encoding="utf-8")
# Backup prima del caricamento del CSV derivato in RAM.
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
shutil.copytree(RUN_DIR, DRIVE_DIR, dirs_exist_ok=True)
print("CSV e report salvati su Drive:", DRIVE_DIR)


## 5. Carica e verifica il log derivato

Ora viene caricato in RAM solo il CSV già filtrato e compresso. Controlla il riepilogo prima di proseguire: la discovery e la valutazione successive useranno **questo `df`**, non il file raw originale.

Se il runtime si interrompe qui o durante la discovery, il CSV e `preprocessing_report.json` restano su Drive. Puoi riutilizzare quel CSV in una nuova esecuzione con `INPUT_KIND="event_log_csv"`.


In [ ]:
df = load_event_log(PREPARED_CSV)
print(f"Log per la discovery: {len(df):,} eventi; {df['case:concept:name'].nunique():,} casi; {df['concept:name'].nunique()} zone")
print("Timestamp CSV:", df["time:timestamp"].min(), "→", df["time:timestamp"].max())
print("Stima deep del DataFrame:", round(df.memory_usage(deep=True).sum() / 2**30, 3), "GiB")
print("RAM ancora disponibile:", round(psutil.virtual_memory().available / 2**30, 2), "GiB")
display(df.head(20))


## 6. Esegui il confronto (questa cella comprende discovery e valutazione)

Non eseguire contemporaneamente altri training nello stesso runtime. Il modello corrente nella cartella del PC non viene modificato.

Se una configurazione supera i limiti, il confronto prosegue con le altre e conserva i modelli già scoperti. Prima di aumentare la RAM del worker controlla la RAM disponibile del runtime. `None` disabilita il limite di RAM, ma aumenta il rischio di perdere il runtime.


In [ ]:
from colab_benchmark import run_comparison, DEFAULT_CONFIGS
CONFIGS = DEFAULT_CONFIGS  # puoi selezionare una parte della lista per una prova più breve
MAX_MODEL_SECONDS = 600
MAX_MODEL_RAM_GB = 4
SEED = 42
# Cartelle nuove per ogni confronto: non mescolano risultati di esecuzioni diverse.
BENCHMARK_ID = datetime.now(timezone.utc).strftime("comparison_%Y%m%dT%H%M%S%fZ")
BENCHMARK_DIR = RUN_DIR / BENCHMARK_ID
BENCHMARK_DRIVE_DIR = DRIVE_DIR / BENCHMARK_ID
print("Backup del confronto:", BENCHMARK_DRIVE_DIR)
# Il supervisore rilegge il CSV una volta per fissare i casi; poi ogni worker libera la propria RAM.
if "df" in globals():
    del df
gc.collect()
comparison = run_comparison(
    PREPARED_CSV, BENCHMARK_DIR, configs=CONFIGS,
    max_eval_cases=MAX_EVAL_CASES, seed=SEED,
    timeout_seconds=MAX_MODEL_SECONDS, max_memory_gb=MAX_MODEL_RAM_GB,
    backup_dir=BENCHMARK_DRIVE_DIR,
)
display(comparison.sort_values("F1", ascending=False, na_position="last"))


## 7. Leggi e scarica il confronto

- Confronta **precision e media armonica**, non solo la fitness.
- Controlla `Differenza ordine`: se la fitness originale e quella mescolata sono quasi uguali, il modello distingue poco l'ordine degli eventi.
- Controlla le etichette omesse: cancellare eventi rari può aumentare un punteggio senza migliorare l'utilità.
- `timeout`, `memory_limit` e `error` non sono metriche basse: sono tentativi non completati. I log sono in `candidates/<id>/worker.log`.

Lo ZIP contiene `comparison.csv`, risultati/manifest JSON e `data/` con CSV comune, tutti i modelli scoperti, metriche e metadati. La dashboard può caricare i tre file CSV/PKL/JSON della nuova cartella `data/` e confrontare i candidati. Le metriche dei candidati non completati sono riportate come errori.

I singoli PKL/PNML sono anche in `candidates/<id>/`. In caso di interruzione del runtime controlla il backup su Drive: ogni modello arrivato alla fine della discovery viene salvato lì prima della precision. Se il confronto non ha creato alcun modello, la dashboard non potrà caricare un dizionario vuoto: consulta prima `comparison.csv` e i log.


In [ ]:
from google.colab import files
archive = package_results(BENCHMARK_DIR)
shutil.copy2(archive, BENCHMARK_DRIVE_DIR / "aruba_comparison.zip")
print("ZIP su Drive:", BENCHMARK_DRIVE_DIR / "aruba_comparison.zip")
files.download(str(archive))
